In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr

In [2]:
llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch_full = xr.open_dataset(llc_path, consolidated=True)
emulator_path = '/orcd/data/abodner/002/cody/inference_patch/predictions/2026-03-26-eval:samudra_llc:epoch1_oct2012_patchloc=i(2880,3600)-j(720,1440)/predictions_4d.zarr'
emulator_patch_full = xr.open_dataset(emulator_path, consolidated=True)
#emulator_patch_full = xr.open_dataset(llc_path, consolidated=True)

In [10]:
extent = 1
shift =10
llc_patch = llc_patch_full.isel(time=slice(9216 + shift, 9216 + shift + extent))
emulator_patch = emulator_patch_full.isel(time=slice(0 + shift, shift+extent))#(9216, 9216 + extent))#(0, extent))

In [11]:
# Rename lat/lon to j/i
emulator_patch = emulator_patch.rename({'lat': 'j', 'lon': 'i'})

# Copy over coordinates from llc_patch
emulator_patch['XC'] = llc_patch['XC']
emulator_patch['YC'] = llc_patch['YC']
emulator_patch['rA'] = llc_patch['rA']
emulator_patch['Z'] = llc_patch['Z']

In [12]:
# LLC Patch
dz_llc = np.diff(llc_patch.Z.values)
dz_llc = dz_llc[np.newaxis, :, np.newaxis, np.newaxis]

du_dz_llc = np.diff(llc_patch['U'].values, axis=1) / dz_llc
dv_dz_llc = np.diff(llc_patch['V'].values, axis=1) / dz_llc

S2_llc = du_dz_llc**2 + dv_dz_llc**2

llc_patch['S2'] = xr.DataArray(S2_llc, dims=['time', 'k', 'j', 'i'], 
                                coords={'time': llc_patch['time'], 'k': llc_patch['k'][:-1],
                                        'j': llc_patch['j'], 'i': llc_patch['i']})

# Emulator Patch
dz_emulator = np.diff(emulator_patch.Z.values)
dz_emulator = dz_emulator[np.newaxis, :, np.newaxis, np.newaxis]

du_dz_emulator = np.diff(emulator_patch['U'].values, axis=1) / dz_emulator
dv_dz_emulator = np.diff(emulator_patch['V'].values, axis=1) / dz_emulator

S2_emulator = du_dz_emulator**2 + dv_dz_emulator**2

emulator_patch['S2'] = xr.DataArray(S2_emulator, dims=['time', 'k', 'j', 'i'], 
                                    coords={'time': emulator_patch['time'], 'k': emulator_patch['k'][:-1],
                                            'j': emulator_patch['j'], 'i': emulator_patch['i']})

In [ ]:
fig, axes = plt.subplots(50, 3, figsize=(15, 150), dpi=100)

for k in range(50):
    llc_s2 = llc_patch.isel(time=0, k=k).S2
    emu_s2 = emulator_patch.isel(time=0, k=k).S2
    
    # Shared vmin/vmax for LLC and Emulator (1st and 99th percentiles)
    vmin = min(float(llc_s2.quantile(0.01)), float(emu_s2.quantile(0.01)))
    vmax = max(float(llc_s2.quantile(0.99)), float(emu_s2.quantile(0.99)))
    
    # LLC in column 0
    cf0 = llc_s2.plot(ax=axes[k, 0], cmap='viridis', vmin=vmin, vmax=vmax, add_colorbar=False)
    axes[k, 0].set_title(f'LLC S2 k={k}', fontsize=8)
    axes[k, 0].set_xlabel('')
    axes[k, 0].set_ylabel('')
    
    # Emulator in column 1
    cf1 = emu_s2.plot(ax=axes[k, 1], cmap='viridis', vmin=vmin, vmax=vmax, add_colorbar=False)
    axes[k, 1].set_title(f'Emulator S2 k={k}', fontsize=8)
    axes[k, 1].set_xlabel('')
    axes[k, 1].set_ylabel('')
    
    # Shared colorbar for columns 0 and 1
    cbar = plt.colorbar(cf1, ax=[axes[k, 0], axes[k, 1]], fraction=0.046, pad=0.04)
    
    # Difference in column 2 (RWB), normalized by min/max
    diff = llc_s2 - emu_s2
    vmax_diff = max(abs(float(diff.quantile(0.01))), abs(float(diff.quantile(0.99))))
    diff_normalized = diff / vmax_diff
    cf2 = diff_normalized.plot(ax=axes[k, 2], cmap='RdBu_r', vmin=-1, vmax=1, add_colorbar=False)
    axes[k, 2].set_title(f'(LLC - Emu) / max k={k}', fontsize=8)
    axes[k, 2].set_xlabel('')
    axes[k, 2].set_ylabel('')
    
    # Colorbar for difference
    plt.colorbar(cf2, ax=axes[k, 2], fraction=0.046, pad=0.04)

#plt.tight_layout()
plt.show()